In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

from itertools import combinations

  

This notebook explores several unsupervised learning techniques including:
- K-Means Clustering (from scratch)
- Hierarchical Clustering
- Principal Component Analysis (PCA)
- t-SNE Visualization
- Association Rule Mining (Apriori)

The main project task is Market Basket Analysis using Apriori to discover product associations.

In [8]:
data = pd.read_csv("/kaggle/input/datasets/heeraldedhia/groceries-dataset/Groceries_dataset.csv")

data.head()

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [10]:
data = pd.DataFrame({"Transaction": transactions})

data.head()

,Transaction
0,"[citrus fruit, semi-finished bread, margarine,..."
1,"[tropical fruit, yogurt, coffee]"
2,[whole milk]
3,"[pip fruit, yogurt, cream cheese , meat spreads]"
4,"[other vegetables, whole milk, condensed milk,..."


In [12]:
transactions = []

for t in data['Transaction']:
    
    if isinstance(t, list):
        transactions.append(t)
        
    else:
        transactions.append([item.strip() for item in str(t).split(',')])

transactions[:5]

[['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups'],
 ['tropical fruit', 'yogurt', 'coffee'],
 ['whole milk'],
 ['pip fruit', 'yogurt', 'cream cheese ', 'meat spreads'],
 ['other vegetables',
  'whole milk',
  'condensed milk',
  'long life bakery product']]

In [13]:
print("Total Transactions:", len(transactions))
print("Example:", transactions[0])

Total Transactions: 9835
Example: ['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups']


In [15]:
def get_support(itemset, transactions):

    count = 0

    for transaction in transactions:
        if set(itemset).issubset(set(transaction)):
            count += 1

    return count / len(transactions)

In [16]:
def apriori(transactions, min_support=0.02):

    items = set()

    for transaction in transactions:
        for item in transaction:
            items.add(item)

    items = [[item] for item in items]

    frequent_itemsets = []

    for item in items:

        support = get_support(item, transactions)

        if support >= min_support:
            frequent_itemsets.append((item, support))

    return frequent_itemsets

In [17]:
frequent_items = apriori(transactions, 0.02)

frequent_items[:10]

[(['brown bread'], 0.06487036095577021),
 (['fruit/vegetable juice'], 0.0722928317234367),
 (['bottled water'], 0.11052364006100661),
 (['coffee'], 0.05805795627859685),
 (['grapes'], 0.022369089984748347),
 (['domestic eggs'], 0.06344687341128623),
 (['butter'], 0.05541433655312659),
 (['sugar'], 0.03385866802236909),
 (['cream cheese '], 0.03721403152008134),
 (['frankfurter'], 0.058973055414336555)]

In [18]:
from itertools import combinations

def generate_rules(frequent_items, transactions, min_confidence=0.3):

    rules = []

    for itemset, support in frequent_items:

        if len(itemset) < 2:
            continue

        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):

                antecedent = list(antecedent)
                consequent = list(set(itemset) - set(antecedent))

                support_a = get_support(antecedent, transactions)

                confidence = support / support_a

                if confidence >= min_confidence:
                    rules.append((antecedent, consequent, confidence))

    return rules

In [19]:
rules = generate_rules(frequent_items, transactions)

rules[:10]

[]